In [0]:
import sys
sys.path.append('./..')

from config.tables.table_config import TABLE_CONFIG
from pyspark.sql import functions as F


In [0]:
transactions_df = spark.table(TABLE_CONFIG.TRASACTIONS)
offers_df = spark.table(TABLE_CONFIG.OFFERS)
profile_df = spark.table(TABLE_CONFIG.PROFILE)
df = spark.table(TABLE_CONFIG.DATA_PROCESSING)

In [0]:
df_result = (
    transactions_df
    .filter(F.col("time_since_test_start").isNotNull())
    .withColumn("day", F.floor(F.col("time_since_test_start")))
    .groupBy("day")
    .agg(
        F.count(F.when(F.col("event") == "transaction", 1)).alias("transactions"),
        F.count(F.when(F.col("event") == "offer received", 1)).alias("offers_received"),
        F.count(F.when(F.col("event") == "offer viewed", 1)).alias("offers_viewed"),
        F.count(F.when(F.col("event") == "offer completed", 1)).alias("offers_completed"),
    )
    .orderBy("day")
)

df_result.display()

Databricks visualization. Run in Databricks to view.

In [0]:
result = (
    transactions_df.alias("t")
    .join(
        offers_df.alias("o"),
        F.col("t.offer_id") == F.col("o.offer_id"),
        "inner",
    )
    .filter(
        (F.col("t.event") == "offer completed")
        & F.col("o.offer_type").isNotNull()
        & F.col("t.time_since_test_start").isNotNull()
    )
    .withColumn("day", F.floor(F.col("t.time_since_test_start")))
    .groupBy(F.col("o.offer_type"), F.col("day"))
    .agg(
        F.count("*").alias("completions"),
        F.sum(F.col("t.reward")).alias("total_rewards"),
    )
    .orderBy("day", F.col("o.offer_type"))
)

result.display()

Databricks visualization. Run in Databricks to view.

In [0]:
result = (
    profile_df
    .filter(
        F.col("gender").isNotNull() &
        F.col("age").isNotNull()
    )
    .withColumn(
        "age_group",
        F.when(F.col("age") < 30, "18-29")
         .when(F.col("age") < 40, "30-39")
         .when(F.col("age") < 50, "40-49")
         .when(F.col("age") < 60, "50-59")
         .when(F.col("age") < 70, "60-69")
         .otherwise("70+")
    )
    .groupBy("gender", "age_group")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.avg("credit_card_limit").alias("avg_credit_limit"),
    )
    .orderBy("gender", "age_group")
)

result.display()

Databricks visualization. Run in Databricks to view.

In [0]:
result = (
    offers_df.alias("o")
    .join(
        transactions_df.alias("t"),
        F.col("o.offer_id") == F.col("t.offer_id"),
        "left"
    )
    .filter(F.col("o.offer_type").isNotNull())
    .groupBy(F.col("o.offer_type"))
    .agg(
        F.countDistinct(
            F.when(
                F.col("t.event") == "offer received",
                F.col("t.account_id"),
            )
        ).alias("received"),
        F.countDistinct(
            F.when(
                F.col("t.event") == "offer viewed",
                F.col("t.account_id"),
            )
        ).alias("viewed"),
        F.countDistinct(
            F.when(
                F.col("t.event") == "offer completed",
                F.col("t.account_id"),
            )
        ).alias("completed"),
    )
    .orderBy(F.desc("received"))
)

result.display()

Databricks visualization. Run in Databricks to view.

In [0]:
result = result.withColumn('%_viewed_rate', 100*F.col('viewed')/F.col('received'))
result = result.withColumn('%_completed_rate', 100*F.col('completed')/F.col('received'))

result.display()

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

# Filter the data
df_filtered = (
    df
    .filter(
        (F.col("offer_id").isNotNull()) &
        (F.col("offer_type") != "informational")
    )
)

# Aggregate by offer_id
offer_perf = (
    df_filtered
    .groupBy("offer_id")
    .agg(
        F.max("offer_type").alias("offer_type"),
        F.max("discount_value").alias("discount_value"),
        F.max("min_value").alias("min_value"),
        F.countDistinct(
            F.when(
                F.col("event") == "offer received",
                F.col("account_id")
            )
        ).alias("customers_received"),
        F.countDistinct(
            F.when(
                F.col("event") == "offer completed",
                F.col("account_id")
            )
        ).alias("customers_completed"),
    )
    .withColumn(
        "completion_rate_pct",
        F.round(
            F.when(
                F.col("customers_received") != 0,
                100.0
                * F.col("customers_completed")
                / F.col("customers_received"),
            ),
            2,
        )
    )
)

# Select the final columns and create the label
result = (
    offer_perf
    .select(
        'offer_id',
        "offer_type",
        "discount_value",
        "min_value",
        "completion_rate_pct"
    )
    .orderBy(F.desc("completion_rate_pct"))
)

# Show the result
result.display()

In [0]:
from pyspark.sql import functions as F

amounts_df = df.groupby('account_id').agg(F.sum('amount').alias('total_amount'))

percentiles = [0, 0.25, 0.5, 0.75, 1]
percentile_values = amounts_df.approxQuantile('total_amount', percentiles, 0.01)

percentile_labels = ['Min', '25th Percentile', 'Median', '75th Percentile', 'Max']
percentile_df = spark.createDataFrame(
    [(label, value) for label, value in zip(percentile_labels, percentile_values)],
    ['Percentile', 'Total Amount']
)

display(percentile_df)

In [0]:
display(df.groupby('account_id').agg(F.sum('amount')))

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

customer_behavior = (
    df
    .filter(F.col("age").isNotNull())
    .groupBy("account_id")
    .agg(
        F.sum(
            F.when(F.col("event") == "transaction", F.col("amount"))
             .otherwise(0)
        ).alias("total_spent")
    )
)

customer_behavior = customer_behavior.withColumn(
    "spending_segment",
    F.when(F.col("total_spent") >= 161, "High Spender")
     .when(F.col("total_spent") >= 90, "Medium Spender")
     .otherwise("Low Spender")
)

customer_offer_performance = (
    customer_behavior.alias("cb")
    .join(
        df.alias("dp"),
        on="account_id",
        how="inner"
    )
    .filter(
        (F.col("dp.offer_id").isNotNull()) &
        (F.col("dp.offer_type").isNotNull())
    )
    .groupBy(
        "account_id",
        "total_spent",
        "spending_segment",
        F.col("dp.offer_type").alias("offer_type")
    )
    .agg(
        F.max(
            F.when(F.col("dp.event") == "offer received", 1)
             .otherwise(0)
        ).alias("received"),
        F.max(
            F.when(F.col("dp.event") == "offer completed", 1)
             .otherwise(0)
        ).alias("completed")
    )
)

result = (
    customer_offer_performance
    .groupBy("spending_segment", "offer_type")
    .agg(
        F.sum("received").alias("total_received"),
        F.sum("completed").alias("total_completed")
    )
    .withColumn(
        "completion_rate_pct",
        F.round(
            F.when(
                F.col("total_received") != 0,
                100.0 * F.col("total_completed") / F.col("total_received")
            ),
            2
        )
    )
    .orderBy("spending_segment", "offer_type")
)

display(result)

Databricks visualization. Run in Databricks to view.